In [1]:
!pip install transformers datasets scikit-learn

import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments

In [2]:
from google.colab import files
uploaded = files.upload()

Saving depression_dataset_reddit_cleaned.csv to depression_dataset_reddit_cleaned.csv


In [3]:
df = pd.read_csv("depression_dataset_reddit_cleaned.csv")

print(df.head())
print(df.columns)

                                          clean_text  is_depression
0  we understand that most people who reply immed...              1
1  welcome to r depression s check in post a plac...              1
2  anyone else instead of sleeping more when depr...              1
3  i ve kind of stuffed around a lot in my life d...              1
4  sleep is my greatest and most comforting escap...              1
Index(['clean_text', 'is_depression'], dtype='object')


In [4]:
df = df.rename(columns={
    "clean_text": "text",
    "is_depression": "label"
})

In [5]:
df = df.dropna()


In [6]:
print(df['label'].value_counts())


label
0    3900
1    3831
Name: count, dtype: int64


In [7]:
#Train-test split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)


In [8]:
#TOKENIZATION
# making use of hugging face transformers
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
class MentalHealthDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MentalHealthDataset(train_encodings, train_labels)
val_dataset = MentalHealthDataset(val_encodings, val_labels)

In [10]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    logging_dir='./logs'
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
!pip install --upgrade transformers

from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 83.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.079097
2,0.118865,0.092854
3,0.039537,0.092681


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1161, training_loss=0.07226076832524052, metrics={'train_runtime': 267.5757, 'train_samples_per_second': 69.334, 'train_steps_per_second': 4.339, 'total_flos': 614383794966528.0, 'train_loss': 0.07226076832524052, 'epoch': 3.0})

In [15]:
preds = trainer.predict(val_dataset)

y_pred = preds.predictions.argmax(axis=1)

print(classification_report(val_labels, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98       783
           1       0.98      0.97      0.97       764

    accuracy                           0.97      1547
   macro avg       0.97      0.97      0.97      1547
weighted avg       0.97      0.97      0.97      1547



In [16]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizer
from sklearn.metrics import accuracy_score, classification_report
import numpy as np


In [17]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


In [18]:
class DistilBERT_BiGRU(nn.Module):
    def __init__(self, hidden_dim=128):
        super(DistilBERT_BiGRU, self).__init__()

        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')

        # 🔥 Freeze BERT
        for param in self.bert.parameters():
            param.requires_grad = False

        self.bigru = nn.GRU(
            input_size=768,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, 2)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():  # extra safety
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        x = outputs.last_hidden_state  # (batch, seq_len, 768)

        x, _ = self.bigru(x)

        # 🔥 FIXED pooling
        x = torch.mean(x, dim=1)

        x = self.dropout(x)
        logits = self.fc(x)

        return logits


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_dataset = TextDataset(train_texts, train_labels, tokenizer)
val_dataset = TextDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

model = DistilBERT_BiGRU().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # 🔥 higher LR


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
from tqdm import tqdm

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss}")


100%|██████████| 387/387 [00:33<00:00, 11.65it/s]


Epoch 1, Loss: 62.69174800021574


100%|██████████| 387/387 [00:30<00:00, 12.82it/s]


Epoch 2, Loss: 39.91406681854278


100%|██████████| 387/387 [00:29<00:00, 13.18it/s]

Epoch 3, Loss: 28.33300808549393


In [21]:
def evaluate_model(model, dataloader):
    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)

            preds = torch.argmax(outputs, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(true_labels, predictions)
    report = classification_report(true_labels, predictions)

    print("Accuracy:", acc)
    print(report)

    print("Prediction Distribution:")
    print(np.unique(predictions, return_counts=True))


In [22]:
evaluate_model(model, val_loader)


Accuracy: 0.9767291531997414
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       783
           1       1.00      0.96      0.98       764

    accuracy                           0.98      1547
   macro avg       0.98      0.98      0.98      1547
weighted avg       0.98      0.98      0.98      1547

Prediction Distribution:
(array([0, 1]), array([813, 734]))


In [23]:
pip install transformers datasets torch scikit-learn


In [24]:
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [25]:
dataset = load_dataset("imdb")

train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [26]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [27]:
from transformers import DistilBertModel
import torch
import torch.nn as nn

class DistilBertBiGRUAttention(nn.Module):
    def __init__(self, num_labels):
        super().__init__()

        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        hidden_size = self.bert.config.hidden_size

        # BiGRU
        self.bigru = nn.GRU(
            input_size=hidden_size,
            hidden_size=128,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Attention
        self.attention = nn.Linear(128 * 2, 1)

        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(128 * 2, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state

        # BiGRU output
        gru_output, _ = self.bigru(sequence_output)

        # Attention
        attn_scores = self.attention(gru_output)
        attn_weights = torch.softmax(attn_scores, dim=1)

        context_vector = torch.sum(attn_weights * gru_output, dim=1)

        context_vector = self.dropout(context_vector)
        logits = self.classifier(context_vector)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}


In [28]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }


In [29]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    logging_steps=50,
    save_strategy="epoch",        # NEW format
    eval_strategy="epoch",        # NEW (not evaluation_strategy)
    load_best_model_at_end=True
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [30]:
model = DistilBertBiGRUAttention(num_labels=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.308109,0.338675,0.856000,0.853955,0.845382,0.862705
2,0.227303,0.494537,0.857000,0.864711,0.803163,0.936475
3,0.019878,0.587300,0.870000,0.872299,0.837736,0.909836


TrainOutput(global_step=750, training_loss=0.24695507256189983, metrics={'train_runtime': 225.4085, 'train_samples_per_second': 26.618, 'train_steps_per_second': 3.327, 'total_flos': 0.0, 'train_loss': 0.24695507256189983, 'epoch': 3.0})

In [32]:
trainer.evaluate()


{'eval_loss': 0.33867475390434265,
 'eval_accuracy': 0.856,
 'eval_f1': 0.8539553752535497,
 'eval_precision': 0.8453815261044176,
 'eval_recall': 0.8627049180327869,
 'eval_runtime': 8.9557,
 'eval_samples_per_second': 111.661,
 'eval_steps_per_second': 13.958,
 'epoch': 3.0}

In [33]:
# Remove duplicates
df.drop_duplicates(subset="text", inplace=True)

# Remove very short texts (noise)
df = df[df['text'].str.len() > 10]

# Reset index
df.reset_index(drop=True, inplace=True)

print(df.shape)


(7620, 2)


In [34]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.1,
    stratify=df['label'],
    random_state=42
)


In [35]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=256)


In [36]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)


In [37]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    dropout=0.3,
    attention_dropout=0.3
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [38]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)


In [39]:
from transformers import Trainer

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=0):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [40]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


In [41]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [42]:
from transformers import EarlyStoppingCallback

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [43]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.150672,0.088339,0.967192,0.966262,0.980822,0.952128
2,0.090474,0.066290,0.976378,0.976000,0.978610,0.973404
3,0.056369,0.067574,0.981627,0.981233,0.989189,0.973404
4,0.046448,0.094290,0.977690,0.977483,0.973615,0.981383


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1716, training_loss=0.10922978398127434, metrics={'train_runtime': 720.8263, 'train_samples_per_second': 38.056, 'train_steps_per_second': 2.381, 'total_flos': 1816922839965696.0, 'train_loss': 0.10922978398127434, 'epoch': 4.0})

In [44]:
trainer.evaluate()


{'eval_loss': 0.06759095191955566,
 'eval_accuracy': 0.9816272965879265,
 'eval_f1': 0.9812332439678284,
 'eval_precision': 0.9891891891891892,
 'eval_recall': 0.973404255319149,
 'eval_runtime': 6.1069,
 'eval_samples_per_second': 124.778,
 'eval_steps_per_second': 7.86,
 'epoch': 4.0}

In [45]:
trainer.save_model("./best_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [46]:
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./best_model/tokenizer_config.json', './best_model/tokenizer.json')

In [48]:
trainer.save_model("mental_health_model")
tokenizer.save_pretrained("mental_health_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('mental_health_model/tokenizer_config.json',
 'mental_health_model/tokenizer.json')

In [50]:
import shutil

shutil.make_archive("mental_health_model", 'zip', "mental_health_model")


'/content/mental_health_model.zip'